# Phase 2 — Offline Cluster Search

Finds Reddit posts from a new dataset that semantically match any of the 44 topic clusters built in Phase 1.

**Folder structure expected:**
```
Phase2/
├── Phase2_Search.ipynb   ← this notebook
├── assets/
│   ├── cluster_centroids.npy   ← from Phase 1 (included)
│   └── cluster_profiles.json   ← from Phase 1 (included)
├── pool/                       ← created by Section A
│   ├── pool_embeddings.npy
│   └── pool_index.csv
└── downloads/                  ← put your Reddit JSONL files here
    ├── r_AskMen_posts.jsonl
    └── ...
```

**Two sections:**
- **Section A — Prep** *(run once)*: cleans and embeds your Reddit JSONL files, saves results to `pool/`
- **Section B — Search** *(run any time)*: pick a cluster, get a matching post instantly

> If `pool/pool_embeddings.npy` is already provided, skip Section A entirely and go straight to Section B.

---
## Section A — Prep  *(run once)*

In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 of Section A — Setup & API Key                          ║
# ║  Paste your OpenAI key below, then run this cell first.         ║
# ╚══════════════════════════════════════════════════════════════════╝

OPENAI_API_KEY = ''   # paste your key here, or leave blank to load from .env

import os, json, re, html, time
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI, AuthenticationError

BASE_DIR      = Path(os.getcwd())
ASSETS_DIR    = BASE_DIR / 'assets'
POOL_DIR      = BASE_DIR / 'pool'
DOWNLOADS_DIR = BASE_DIR / 'downloads'

POOL_DIR.mkdir(exist_ok=True)

EMBED_MODEL = 'text-embedding-3-small'
BATCH_SIZE  = 500
MAX_RETRIES = 5
MIN_WORDS   = 50
MAX_WORDS   = 300

KEEP_COLS = [
    'id', 'subreddit', 'author', 'title', 'selftext',
    'score', 'upvote_ratio', 'num_comments', 'created_utc',
    'over_18', 'is_self', 'stickied', 'locked',
    'removed_by_category', 'permalink',
]
DELETED_AUTHORS = {'[deleted]', 'AutoModerator'}
REMOVED_BODIES  = {'[removed]', '[deleted]', ''}

if not OPENAI_API_KEY:
    try:
        from dotenv import load_dotenv
        load_dotenv(BASE_DIR / '.env', override=True)
        load_dotenv(BASE_DIR.parent / '.env', override=False)
    except ImportError:
        pass
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    raise ValueError('No OpenAI API key found. Paste it into OPENAI_API_KEY above.')

oai = OpenAI(api_key=OPENAI_API_KEY)
oai.embeddings.create(model=EMBED_MODEL, input=['test'])
print('OpenAI key OK ✓')
print(f'Downloads: {DOWNLOADS_DIR}  ({len(list(DOWNLOADS_DIR.glob("*.jsonl")))} JSONL files found)')
print(f'Pool dir:  {POOL_DIR}')

OpenAI key OK ✓
Downloads: /Users/felix/Documents/Phase2/downloads  (0 JSONL files found)
Pool dir:  /Users/felix/Documents/Phase2/pool


In [4]:
# Load all JSONL files from downloads/

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    df = pd.DataFrame(records)
    for c in KEEP_COLS:
        if c not in df.columns:
            df[c] = None
    return df[KEEP_COLS]

files = sorted(DOWNLOADS_DIR.glob('*.jsonl'))
if not files:
    raise FileNotFoundError(f'No JSONL files found in {DOWNLOADS_DIR}')

frames = []
for path in files:
    df = load_jsonl(path)
    frames.append(df)
    print(f'  {path.name}: {len(df):,} rows')

raw = pd.concat(frames, ignore_index=True)
print(f'\nTotal loaded: {len(raw):,} posts  |  {raw["subreddit"].nunique()} subreddits')

  r_AskMen_posts.jsonl: 62,760 rows
  r_AskOldPeople_posts.jsonl: 6,231 rows
  r_AskWomen_posts.jsonl: 27,582 rows
  r_BenignExistence_posts.jsonl: 1,992 rows
  r_CasualUK_posts.jsonl: 12,346 rows
  r_CongratsLikeImFive_posts.jsonl: 1,839 rows
  r_DadForAMinute_posts.jsonl: 1,854 rows
  r_Journaling_posts.jsonl: 6,749 rows
  r_MadeMeSmile_posts.jsonl: 20,856 rows
  r_Mindfulness_posts.jsonl: 1,779 rows
  r_MomForAMinute_posts.jsonl: 5,243 rows
  r_OlderThanYouThinkIAm_posts.jsonl: 216 rows
  r_PointlessStories_posts.jsonl: 2,269 rows
  r_RBI_posts.jsonl: 1,977 rows
  r_TodayIAmHappy_posts.jsonl: 17 rows
  r_Wellthatsucks_posts.jsonl: 6,240 rows
  r_decidingtobebetter_posts.jsonl: 6,861 rows
  r_internetparents_posts.jsonl: 3,842 rows
  r_relationship_advice_posts.jsonl: 154,724 rows
  r_retailhell_posts.jsonl: 3,987 rows
  r_self_posts.jsonl: 27,156 rows
  r_selfimprovement_posts.jsonl: 15,607 rows
  r_stories_posts.jsonl: 5,886 rows
  r_wholesome_posts.jsonl: 2,671 rows

Total loaded:

In [5]:
# 8-step filter — same as Phase 1, no subreddit balancing

def word_count(text: str) -> int:
    return len(text.split())

df = raw.copy()
n0 = len(df)

df = df[df['is_self'] == True]
print(f'After is_self:           {len(df):>7,}  (dropped {n0-len(df):,})')

n = len(df); df = df[df['stickied'] == False]
print(f'After stickied:          {len(df):>7,}  (dropped {n-len(df):,})')

n = len(df); df = df[df['over_18'] == False]
print(f'After NSFW:              {len(df):>7,}  (dropped {n-len(df):,})')

n = len(df)
df = df[df['removed_by_category'].isna()]
df = df[~df['selftext'].isin(REMOVED_BODIES)]
print(f'After removed/deleted:   {len(df):>7,}  (dropped {n-len(df):,})')

n = len(df); df = df[df['score'].fillna(0) >= 1]
print(f'After score floor:       {len(df):>7,}  (dropped {n-len(df):,})')

n = len(df); df = df[~df['author'].isin(DELETED_AUTHORS)]
print(f'After author filter:     {len(df):>7,}  (dropped {n-len(df):,})')

n = len(df)
df['word_count'] = df['selftext'].fillna('').apply(word_count)
df = df[(df['word_count'] >= MIN_WORDS) & (df['word_count'] <= MAX_WORDS)]
print(f'After word count filter: {len(df):>7,}  (dropped {n-len(df):,})')

n = len(df); df = df.drop_duplicates(subset='id')
print(f'After dedup:             {len(df):>7,}  (dropped {n-len(df):,})')

print(f'\nClean pool: {len(df):,} posts  ({len(df)/n0*100:.1f}% of raw)')

After is_self:           340,096  (dropped 40,588)
After stickied:          340,044  (dropped 52)
After NSFW:              300,149  (dropped 39,895)
After removed/deleted:   140,052  (dropped 160,097)
After score floor:       100,290  (dropped 39,762)
After author filter:     100,076  (dropped 214)
After word count filter:  51,081  (dropped 48,995)
After dedup:              51,081  (dropped 0)

Clean pool: 51,081 posts  (13.4% of raw)


In [6]:
# Text cleaning — same function as Phase 1

def clean_text(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    text = re.sub(r'`[^`]*`', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)
    text = re.sub(r'\*{1,3}([^*\n]+)\*{1,3}', r'\1', text)
    text = re.sub(r'_{1,3}([^_\n]+)_{1,3}', r'\1', text)
    text = re.sub(r'~~([^~]+)~~', r'\1', text)
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'^>\s?', '', text, flags=re.MULTILINE)
    text = re.sub(r'^[-*_]{3,}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

df = df.copy()
df['selftext_clean'] = df['selftext'].fillna('').apply(clean_text)
df['title_clean']    = df['title'].fillna('').apply(clean_text)
df['embed_text']     = df['title_clean'] + '\n\n' + df['selftext_clean']
df['url']            = 'https://www.reddit.com' + df['permalink'].fillna('')

print(f'Text cleaned for {len(df):,} posts')

Text cleaned for 51,081 posts


In [7]:
# Embed the pool — skipped automatically if pool_embeddings.npy already exists

pool_emb_path   = POOL_DIR / 'pool_embeddings.npy'
pool_index_path = POOL_DIR / 'pool_index.csv'

if pool_emb_path.exists():
    print('pool_embeddings.npy already exists — loading from disk (skipping API calls)')
    pool_emb   = np.load(pool_emb_path)
    pool_index = pd.read_csv(pool_index_path)
    print(f'Loaded: {pool_emb.shape}  |  {len(pool_index):,} posts')
else:
    def embed_batch(texts):
        for attempt in range(MAX_RETRIES):
            try:
                resp = oai.embeddings.create(model=EMBED_MODEL, input=texts)
                return [item.embedding for item in resp.data]
            except AuthenticationError:
                raise
            except Exception as e:
                wait = 2 ** attempt
                print(f'  ⚠ attempt {attempt+1} failed: {e} — retrying in {wait}s')
                time.sleep(wait)
        raise RuntimeError('Embedding failed after max retries')

    texts = df['embed_text'].tolist()
    n     = len(texts)
    all_vecs = []

    avg_tokens = 150
    est_cost   = n * avg_tokens / 1_000_000 * 0.02
    print(f'Embedding {n:,} posts  |  est. cost: ~${est_cost:.3f}  |  est. time: ~{n//500*3//60+1} min')
    print()

    t0 = time.time()
    for start in range(0, n, BATCH_SIZE):
        batch = texts[start : start + BATCH_SIZE]
        all_vecs.extend(embed_batch(batch))
        done = start + len(batch)
        print(f'  [{done:>6,}/{n:,}]  {done/n*100:.1f}%  elapsed: {time.time()-t0:.0f}s')

    pool_emb = np.array(all_vecs, dtype=np.float32)

    pool_index = df[['id','subreddit','author','title',
                     'selftext_clean','word_count','url']].reset_index(drop=True)

    np.save(pool_emb_path, pool_emb)
    pool_index.to_csv(pool_index_path, index=False)

    print(f'\nDone — shape: {pool_emb.shape}')
    print(f'pool_embeddings.npy → {pool_emb_path}')
    print(f'pool_index.csv      → {pool_index_path}')

Embedding 51,081 posts  |  est. cost: ~$0.153  |  est. time: ~6 min

  [   500/51,081]  1.0%  elapsed: 5s
  [ 1,000/51,081]  2.0%  elapsed: 8s
  [ 1,500/51,081]  2.9%  elapsed: 12s
  [ 2,000/51,081]  3.9%  elapsed: 14s
  [ 2,500/51,081]  4.9%  elapsed: 18s
  [ 3,000/51,081]  5.9%  elapsed: 21s
  [ 3,500/51,081]  6.9%  elapsed: 625s
  [ 4,000/51,081]  7.8%  elapsed: 628s
  [ 4,500/51,081]  8.8%  elapsed: 632s
  [ 5,000/51,081]  9.8%  elapsed: 635s
  [ 5,500/51,081]  10.8%  elapsed: 638s
  [ 6,000/51,081]  11.7%  elapsed: 641s
  [ 6,500/51,081]  12.7%  elapsed: 645s
  [ 7,000/51,081]  13.7%  elapsed: 647s
  [ 7,500/51,081]  14.7%  elapsed: 650s
  [ 8,000/51,081]  15.7%  elapsed: 654s
  [ 8,500/51,081]  16.6%  elapsed: 657s
  [ 9,000/51,081]  17.6%  elapsed: 660s
  [ 9,500/51,081]  18.6%  elapsed: 663s
  [10,000/51,081]  19.6%  elapsed: 667s
  [10,500/51,081]  20.6%  elapsed: 670s
  [11,000/51,081]  21.5%  elapsed: 673s
  [11,500/51,081]  22.5%  elapsed: 677s
  [12,000/51,081]  23.5%  ela

---
## Section B — Search  *(run any time)*

Start here if `pool/pool_embeddings.npy` already exists.  
No API key needed for this section.

In [ ]:
import os, json, textwrap
import numpy as np
import pandas as pd
from pathlib import Path

NEW_DIR    = Path('/home/felix/Simulation/reDo_For_0626/newData')
PHASE1_DIR = Path('/home/felix/Simulation/reDo_For_0626/cleaned')
SIM_THRESH = None  # None = use each cluster's calibrated threshold
                   # Set to a float e.g. 0.50 to override

# Load the pool
pool_emb   = np.load(NEW_DIR / 'pool_embeddings.npy')
pool_index = pd.read_csv(NEW_DIR / 'pool_index.csv')

# Load Phase 1 cluster assets
centroids = np.load(PHASE1_DIR / 'cluster_centroids.npy')
with open(PHASE1_DIR / 'cluster_profiles.json') as f:
    profiles = json.load(f)

# Show cluster menu
rows = [
    {'id': int(k), 'topic_label': v['topic_label'],
     'n_posts': v['n_posts'], 'threshold': v['sim_threshold'],
     'top_keywords': ', '.join(v['top_keywords'][:5])}
    for k, v in sorted(profiles.items(), key=lambda x: -x[1]['n_posts'])
]
df_menu = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.max_rows', 50)
print(f'Pool: {len(pool_emb):,} posts  |  Clusters: {len(profiles)}')
print()
print(df_menu.to_string(index=False))

Pool: 51,081 posts  |  Clusters: 44  |  Centroids: (44, 1536)



' id                  topic_label  n_posts  threshold                                      top_keywords\n  0            Retail Experience     1245     0.4500         store, customer, customers, manager, work\n  1         Mindfulness Practice     1244     0.4500     mindfulness, thoughts, mind, meditation, like\n  2         Journaling Practices      998     0.5049        journal, journaling, write, writing, pages\n  3                Parental Loss      613     0.5242                        dad, im, mom, father, just\n  4         Personal Achievement      529     0.4800                    today, ive, proud, finally, im\n  5           Career Uncertainty      481     0.5139                         job, im, life, dont, feel\n  6      Account Security Issues      431     0.4500              account, video, number, videos, help\n  7              Self Acceptance      351     0.5564                      people, feel, im, dont, like\n  8            Everyday Kindness      235     0.4874           

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 of Section B — Pick a Cluster                           ║
# ║  Change SELECTED_CLUSTER_ID to any id from the table above,     ║
# ║  then run this cell.                                            ║
# ╚══════════════════════════════════════════════════════════════════╝

SELECTED_CLUSTER_ID = 9   # <-- change this number

profile  = profiles[str(SELECTED_CLUSTER_ID)]
centroid = centroids[profile['centroid_row']]
active_threshold = SIM_THRESH if SIM_THRESH is not None else profile['sim_threshold']

# Score all pool posts against this cluster's centroid
norms = np.linalg.norm(pool_emb, axis=1) * np.linalg.norm(centroid)
sims  = (pool_emb @ centroid) / (norms + 1e-10)

mask    = sims >= active_threshold
matches = pool_index[mask].copy()
matches['similarity'] = sims[mask].round(4)
matches = matches.sort_values('similarity', ascending=False).reset_index(drop=True)

print(f"Cluster #{SELECTED_CLUSTER_ID}: '{profile['topic_label']}'")
print(f"Threshold:  {active_threshold:.4f}  (mean={profile['sim_mean']:.4f}  std={profile['sim_std']:.4f})")
print(f"Matches:    {len(matches)} / {len(pool_emb):,} pool posts")
print()

if len(matches):
    # Preview table — top 15, title truncated for readability
    display_df = matches.head(15)[['similarity', 'subreddit', 'title', 'url']].copy()
    display_df.columns = ['Sim', 'Subreddit', 'Title', 'URL']
    display_df['Subreddit'] = 'r/' + display_df['Subreddit']
    display_df['Title'] = display_df['Title'].str[:55]
    pd.set_option('display.max_colwidth', 100)
    pd.set_option('display.max_rows', 20)
    display(display_df)

    # Save ALL matches as TSV — tabs never appear in titles, so columns stay aligned
    tsv_path = NEW_DIR / f'matches_cluster_{SELECTED_CLUSTER_ID}.tsv'
    matches[['similarity', 'subreddit', 'title', 'url', 'word_count']].to_csv(
        tsv_path, sep='\t', index=False
    )
    print(f'\nAll {len(matches)} matches saved → {tsv_path.name}  (open in Excel or Google Sheets)')
else:
    print('No matches found.')
    print('Try: set SIM_THRESH = 0.50 in Cell 1 of Section B and re-run from there.')

Cluster #1: 'Mindfulness Practice'
Threshold:  0.4500  (mean=0.5734  std=0.0963)
Matches:    12088 / 51,081 pool posts

     Sim  Subreddit             Title
  ----------------------------------------------------------------------
  0.7798  r/DecidingToBeBetter  I Feel Stuck in My Mind and Out of Touch With Life
  0.7647  r/Mindfulness         Seeking Guidance: How Can I Begin My Journey into 
  0.7606  r/selfimprovement     The 60-Second Habit That Broke My Mental Chains
  0.7542  r/Mindfulness         Some reflections on my mindfulness journey... And 
  0.7480  r/Mindfulness         Is there an Art to Not Thinking?
  0.7426  r/DecidingToBeBetter  This Diwali, I Realized the ONE Thing That Truly M
  0.7375  r/selfimprovement     BreakingFree from the Cycle of More: How Mindfulne
  0.7341  r/selfimprovement     The 1 Question That Flipped My Self-Awareness Upsi
  0.7300  r/Mindfulness         Reframing outlook on time
  0.7288  r/self                self reflection
  0.7283  r/Mindfuln

In [13]:
# Display a result — picks randomly from top 5 matches
# Re-run this cell alone to get a different post from the same cluster

TOP_N = 5

if len(matches) == 0:
    print('No matches — adjust SIM_THRESH or pick a different cluster')
else:
    pool_size = min(TOP_N, len(matches))
    chosen    = matches.iloc[np.random.default_rng().integers(pool_size)]

    div = '─' * 72
    print(div)
    print(f"CLUSTER #{SELECTED_CLUSTER_ID} — {profile['topic_label'].upper()}")
    print(f"Similarity: {chosen['similarity']:.4f}  "
          f"(threshold {active_threshold:.4f}  |  cluster mean {profile['sim_mean']:.4f})")
    print(div)
    print(f"r/{chosen['subreddit']}  |  {chosen['word_count']} words")
    print(chosen['url'])
    print()
    print(f"TITLE: {chosen['title']}")
    print()
    print(textwrap.fill(str(chosen['selftext_clean']), width=80))

────────────────────────────────────────────────────────────────────────
CLUSTER #1 — MINDFULNESS PRACTICE
Similarity: 0.7798  (threshold 0.4500  |  cluster mean 0.5734)
────────────────────────────────────────────────────────────────────────
r/DecidingToBeBetter  |  258 words
https://www.reddit.com/r/DecidingToBeBetter/comments/1i9ladu/i_feel_stuck_in_my_mind_and_out_of_touch_with_life/

TITLE: I Feel Stuck in My Mind and Out of Touch With Life

I overthink everything. I care too much about things that probably don’t even
matter. My brain is always working overtime, analyzing outcomes, jumping to
conclusions, and running through endless “what if” scenarios. It’s exhausting,
and it makes me feel like I’ve never truly been able to participate in life.
Instead of living in the moment, I retreat into this false reality I’ve built in
my mind. I create fake scenarios, rehearse conversations that will never happen,
and play out alternate versions of reality. It feels like my escape, but I kn

---
## Quick Reference

| Variable | Where | Default | Effect |
|---|---|---|---|
| `OPENAI_API_KEY` | `ph2-config` | `''` | Required for Section A only |
| `SELECTED_CLUSTER_ID` | `ph2-pick` | `9` | Which cluster to search (0–43) |
| `SIM_THRESH` | `ph2-search-setup` | `None` | `None` = per-cluster threshold. Float = global override |
| `TOP_N` | `ph2-display` | `5` | How many top matches to randomly pick from |

**Different cluster** → change `SELECTED_CLUSTER_ID`, re-run `ph2-pick` + `ph2-display`  
**Different post, same cluster** → re-run `ph2-display` only  
**0 matches** → set `SIM_THRESH = 0.50` in `ph2-search-setup`, re-run from there